# Obstacle recognition v2: people, pallets, carts, forklifts (YOLO11n boxes)

Images from the robot's colour recognition camera in the simulated warehouse (640 x 480, 20 cm above the floor), labelled automatically by Gazebo's bounding-box camera: **person, pallet, cart, forklift** (pallet jacks count as forklifts). About 15 % of the images show no object at all (racks, walls, station legs), so the model learns what is *not* an obstacle.

**v2** (after v1 found an unseen wooden pallet only 6 % of the time): pallets built from parts in 40 variants (block, stringer, plastic; random size, colour, load), recoloured carts, forklifts and pallets, five more carts, and unlabelled distractors (boxes, cones, barrels, bins, crates). Same training settings as v1; only the data changed. The held-out test set is the **same v1 test** (`obstacles_v1_test.zip`, already in Drive), so v1 and v2 are compared on the same images.

Against overfitting: the held-out test set uses **people, pallet and cart models never seen in training**, another seed and other places; nano model, early stopping; train vs validation losses plotted.

**Steps**: Runtime > Change runtime type > **T4 GPU**; run the cells in order; upload `obstacles_v2.zip` (the test zip is already in Drive from v1) (they are kept in Google Drive). If the session resets, run cells 1 and 2, then cell 6 (resume). The last cell downloads `obstacle_detector_v2.zip`; put it in the workspace's `ai_data/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40 onnx onnxslim onnxscript

In [ ]:
# Save the training to Google Drive: a disconnect then loses nothing.
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
import os; os.makedirs(RUNS, exist_ok=True)

In [ ]:
import os, shutil
from google.colab import files
DATA = '/content/drive/MyDrive/tugger_data'
os.makedirs(DATA, exist_ok=True)
NEEDED = ['obstacles_v2.zip', 'obstacles_v1_test.zip']
missing = [z for z in NEEDED if not os.path.exists(f'{DATA}/{z}')]
if missing:
    print('Upload:', missing)
    for z in files.upload():
        shutil.move(z, f'{DATA}/{z}')
for z in NEEDED:
    !unzip -q -o {DATA}/{z} -d /content/
!ls /content/obstacles_v2/images/train | wc -l; ls /content/obstacles_v2/images/val | wc -l; ls /content/obstacles_v1_test/images/test | wc -l

In [ ]:
import yaml
cfg = yaml.safe_load(open('/content/obstacles_v2/data.yaml'))
cfg['path'] = '/content/obstacles_v2'
yaml.safe_dump(cfg, open('/content/obstacles_v2/data_colab.yaml', 'w'))
test = dict(cfg, path='/content/obstacles_v1_test', train='images/test', val='images/test')
yaml.safe_dump(test, open('/content/obstacles_v1_test/data_test.yaml', 'w'))
cfg

In [ ]:
import numpy as np
np.trapz = np.trapezoid   # numpy 2 renamed it; Ultralytics 8.3.40 still calls the old name
from ultralytics import YOLO
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
model = YOLO('yolo11n.pt')
model.train(data='/content/obstacles_v2/data_colab.yaml', imgsz=640, epochs=100, batch=32,
            patience=20, fliplr=0.5, flipud=0.0, degrees=0.0, mosaic=1.0, scale=0.5,
            translate=0.1, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, weight_decay=0.0005,
            project=RUNS, name='obstacles_v2', seed=11, deterministic=True)

In [ ]:
# ONLY if the training stopped (disconnect, restart): run cells 1, 2 (Drive) and this one;
# it continues from the last finished epoch.
import numpy as np
np.trapz = np.trapezoid
from ultralytics import YOLO
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
YOLO(f'{RUNS}/obstacles_v2/weights/last.pt').train(resume=True)

In [ ]:
# Overfitting check: training and validation losses side by side.
import pandas as pd, matplotlib.pyplot as plt
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
r = pd.read_csv(f'{RUNS}/obstacles_v2/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['box_loss', 'cls_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/obstacles_v2/overfitting_check.png', dpi=150); plt.show()

In [ ]:
import numpy as np, json
np.trapz = np.trapezoid
from ultralytics import YOLO
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
best = YOLO(f'{RUNS}/obstacles_v2/weights/best.pt')
val = best.val(data='/content/obstacles_v2/data_colab.yaml', imgsz=640, project=RUNS, name='obstacles_v2_val')
test = best.val(data='/content/obstacles_v1_test/data_test.yaml', imgsz=640, project=RUNS, name='obstacles_v2_heldout_test')
summary = {'val_mAP50-95': float(val.box.map), 'val_mAP50': float(val.box.map50),
           'test_mAP50-95': float(test.box.map), 'test_mAP50': float(test.box.map50),
           'test_per_class_mAP50-95': dict(zip(test.names.values(), map(float, test.box.maps)))}
print(summary)
json.dump(summary, open(f'{RUNS}/obstacles_v2/summary.json', 'w'), indent=1)
# The ONNX file is made on the laptop (single file, for OpenCV).

In [ ]:
RUNS = '/content/drive/MyDrive/tugger_runs'   # set again here: works after a restart too
!cd {RUNS} && zip -q -r /content/obstacle_detector_v2.zip obstacles_v2 obstacles_v2_val obstacles_v2_heldout_test -x '*/weights/last.pt'
from google.colab import files
files.download('/content/obstacle_detector_v2.zip')